In [1]:
import pandas as pd

df = pd.read_csv("heart_disease_uci.csv")

print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nMissing values:\n", df.isnull().sum())
print("\nFirst 5 rows:\n", df.head())

Shape: (920, 16)

Columns:
 Index(['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs',
       'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='object')

Missing values:
 id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

First 5 rows:
    id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slo

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Create target
df['target'] = df['num'].apply(lambda x: 1 if x > 0 else 0)
df.drop(['num', 'id'], axis=1, inplace=True)

# Separate features
X = df.drop('target', axis=1)
y = df['target']

# Split numerical and categorical
num_cols = X.select_dtypes(include=['float64', 'int64']).columns
cat_cols = X.select_dtypes(include=['object', 'bool']).columns

# Impute numerical
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])

# Impute categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Preprocessing complete ✅")
print("Train shape:", X_train.shape)

Preprocessing complete ✅
Train shape: (736, 21)


In [4]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=200, random_state=42)
svm = SVC(probability=True, random_state=42)
lr = LogisticRegression(max_iter=1000)

In [5]:
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)
lr.fit(X_train, y_train)

print("RF:", accuracy_score(y_test, rf.predict(X_test)))
print("SVM:", accuracy_score(y_test, svm.predict(X_test)))
print("LR:", accuracy_score(y_test, lr.predict(X_test)))

RF: 0.842391304347826
SVM: 0.8586956521739131
LR: 0.8315217391304348


In [6]:
voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('svm', svm),
        ('lr', lr)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)
print("Voting Accuracy:", accuracy_score(y_test, voting.predict(X_test)))

Voting Accuracy: 0.8478260869565217


In [7]:
stacking = StackingClassifier(
    estimators=[
        ('rf', rf),
        ('svm', svm),
        ('lr', lr)
    ],
    final_estimator=LogisticRegression()
)

stacking.fit(X_train, y_train)
print("Stacking Accuracy:", accuracy_score(y_test, stacking.predict(X_test)))

Stacking Accuracy: 0.8369565217391305


In [8]:
import joblib

joblib.dump(stacking, "heart_stacking_model.pkl")
joblib.dump(scaler, "heart_scaler.pkl")
joblib.dump(X.columns, "heart_columns.pkl")

['heart_columns.pkl']